<a href="https://colab.research.google.com/github/Adlite07/nlp-financial-risk-early-warning/blob/main/notebooks/01_stage1_news_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path                # Path makes file paths safe across operating systems

# Root folder for the whole project, inside your Drive
ROOT = Path('/content/drive/MyDrive/risk-nlp-project')

# One folder per purpose keeps the project tidy, and mirrors the GitHub repo layout later
for sub in ['data/raw', 'data/processed', 'models', 'results', 'notebooks', 'src']:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)   # exist_ok=True: no error if it already exists

print(sorted(p.relative_to(ROOT).as_posix() for p in ROOT.rglob('*')))  # list what we made

['data', 'data/processed', 'data/raw', 'data/raw/phrasebank_50agree.csv', 'models', 'notebooks', 'results', 'results/environment.json', 'src']


In [ ]:
import pandas as pd

# Path to raw data folder
RAW_DATA_DIR = ROOT / 'data/raw'

# List all files currently inside the raw data directory
raw_files = list(RAW_DATA_DIR.glob('*'))
print(f"Files in raw directory: {[f.name for f in raw_files]}")

# Helper Template: Uncomment and modify the lines below to load your specific dataset when uploaded
# if raw_files:
#     # Example for loading the first CSV found in your raw directory
#     csv_files = list(RAW_DATA_DIR.glob('*.csv'))
#     if csv_files:
#         df = pd.read_csv(csv_files[0])
#         display(df.head())
# else:
#     print("Please place your dataset files inside:", RAW_DATA_DIR)

In [3]:
# "<4.0" means any version below 4.0, which still allows dataset scripts
!pip install -q "datasets<4.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


In [4]:
import sys, torch, sklearn, pandas, numpy
import transformers, datasets

# Record the exact versions: different versions can give slightly different results
print('Python      :', sys.version.split()[0])
print('torch       :', torch.__version__)
print('transformers:', transformers.__version__)
print('datasets    :', datasets.__version__)
print('sklearn     :', sklearn.__version__)
print('pandas      :', pandas.__version__)

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name     :', torch.cuda.get_device_name(0))

Python      : 3.13.16
torch       : 2.14.1+cu130
transformers: 5.22.1
datasets    : 3.6.0
sklearn     : 1.6.1
pandas      : 2.2.3
GPU available: True
GPU name     : Tesla T4


In [5]:
import random, numpy as np, torch

SEED = 42   # any number works; what matters is that you never change it between runs

def set_seed(seed: int = SEED):
    random.seed(seed)                    # Python's built-in random module
    np.random.seed(seed)                 # NumPy (used by scikit-learn too)
    torch.manual_seed(seed)              # PyTorch on CPU
    torch.cuda.manual_seed_all(seed)     # PyTorch on all GPUs

set_seed()
print(np.random.rand(3))   # run this cell twice: you should get identical numbers each time

[0.37454012 0.95071431 0.73199394]


In [6]:
import json, platform

env = {
    'seed': SEED,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'sklearn': sklearn.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
with open(ROOT / 'results' / 'environment.json', 'w') as f:
    json.dump(env, f, indent=2)          # indent=2 makes the file readable
print('Saved to', ROOT / 'results' / 'environment.json')

Saved to /content/drive/MyDrive/risk-nlp-project/results/environment.json


In [7]:
import datasets
print(datasets.__version__)   # should print a number starting with 3

3.6.0


In [8]:
from huggingface_hub import list_repo_files

# Ask the Hub for the file list of the namespaced repo. This does not download the data.
try:
    files = list_repo_files("takala/financial_phrasebank", repo_type="dataset")
    for f in files:
        print(f)
except Exception as e:
    print("LIST FAILED:", type(e).__name__, str(e)[:300])

.gitattributes
README.md
data/FinancialPhraseBank-v1.0.zip
financial_phrasebank.py


In [9]:
from huggingface_hub import hf_hub_download
import zipfile

# Download the zip from the dataset repo. The filename is exactly as in your listing.
zip_path = hf_hub_download(
    repo_id="takala/financial_phrasebank",
    filename="data/FinancialPhraseBank-v1.0.zip",
    repo_type="dataset",
)
print("Downloaded to:", zip_path)

# Look inside the zip without extracting everything
with zipfile.ZipFile(zip_path) as z:
    for name in z.namelist():
        print(name)

data/FinancialPhraseBank-v1.0.zip: reconstructing file:   0%|          |  0.00B /  682kB            

data/FinancialPhraseBank-v1.0.zip: downloading bytes:           |  0.00B            

Downloaded to: /root/.cache/huggingface/hub/datasets--takala--financial_phrasebank/snapshots/8d3fe0c36d5feec6b3cc5e455b0fcb4820fb9964/data/FinancialPhraseBank-v1.0.zip
FinancialPhraseBank-v1.0/
FinancialPhraseBank-v1.0/License.txt
__MACOSX/
__MACOSX/FinancialPhraseBank-v1.0/
__MACOSX/FinancialPhraseBank-v1.0/._License.txt
FinancialPhraseBank-v1.0/README.txt
__MACOSX/FinancialPhraseBank-v1.0/._README.txt
FinancialPhraseBank-v1.0/Sentences_50Agree.txt
FinancialPhraseBank-v1.0/Sentences_66Agree.txt
FinancialPhraseBank-v1.0/Sentences_75Agree.txt
FinancialPhraseBank-v1.0/Sentences_AllAgree.txt


In [10]:
# Paste the exact path of the 50%-agreement file from the Cell 1 output
TARGET = "FinancialPhraseBank-v1.0/Sentences_50Agree.txt"   # CHECK this against your listing

with zipfile.ZipFile(zip_path) as z:
    raw = z.read(TARGET)                      # read the file as raw bytes

# Try UTF-8 first; fall back to latin-1 (an older encoding) if that fails
try:
    text = raw.decode("utf-8")
    print("Decoded as utf-8")
except UnicodeDecodeError:
    text = raw.decode("latin-1")
    print("Decoded as latin-1")

lines = text.splitlines()                      # one sentence per line
print("Number of lines:", len(lines))
for line in lines[:5]:                         # show the first five lines exactly as they are
    print(repr(line))                          # repr shows hidden characters

Decoded as latin-1
Number of lines: 4846
'According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .@neutral'
'Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said .@neutral'
'The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .@negative'
'With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .@positive'
"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating 

In [11]:
import pandas as pd

rows = []
for line in lines:
    line = line.strip()
    if not line:
        continue                                    # skip blank lines
    # rsplit from the right, once: a sentence might contain '@' itself,
    # but the label is always after the LAST '@'
    sentence, label_word = line.rsplit("@", 1)
    rows.append((sentence.strip(), label_word.strip().lower()))

df = pd.DataFrame(rows, columns=["sentence", "label_name"])

# Check that only the three expected labels appear (catches parsing mistakes)
print(df["label_name"].value_counts())

# Our own numeric labels, fixed here so they never change later
label_names = ["negative", "neutral", "positive"]
df["label"] = df["label_name"].map({n: i for i, n in enumerate(label_names)})
assert df["label"].notna().all(), "Unexpected label word found"   # stops if parsing went wrong
print(df.shape)

label_name
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64
(4846, 3)


In [12]:
# Class balance as percentages
print((df["label_name"].value_counts() / len(df) * 100).round(1))

# Sentence length in words
df["n_words"] = df["sentence"].str.split().str.len()
print(df["n_words"].describe())

# Duplicates: identical sentences could end up in both train and test (leakage)
print("Duplicate sentences:", df["sentence"].duplicated().sum())

# Three random examples per class: read them, numbers alone can mislead
for lab in label_names:
    print(f"\n--- {lab} ---")
    for s in df[df["label_name"] == lab]["sentence"].sample(3, random_state=42):
        print("*", s)

# Save the raw table to Drive so we never depend on the loader again
df[["sentence", "label", "label_name"]].to_csv(
    ROOT / "data" / "raw" / "phrasebank_50agree.csv", index=False)
print("Saved")

label_name
neutral     59.4
positive    28.1
negative    12.5
Name: count, dtype: float64
count    4846.000000
mean       23.101114
std         9.958474
min         2.000000
25%        16.000000
50%        21.000000
75%        29.000000
max        81.000000
Name: n_words, dtype: float64
Duplicate sentences: 8

--- negative ---
* The company decided at the end of 2008 to temporarily shut down its ammonia plant in Billingham and extend the maintenance period at its Ince facility .
* down to EUR5 .9 m H1 '09 3 August 2009 - Finnish media group Ilkka-Yhtyma Oyj ( HEL : ILK2S ) said today its net profit fell 45 % on the year to EUR5 .9 m in the first half of 2009 .
* The steelmaker said that the drop in profit was explained by the continuing economic uncertainty , mixed with the current drought in bank lending , resulting in a decline in demand for its products as customers find it increasingly difficult to fund operations .

--- neutral ---
* The center offers a comprehensive range of devi

In [13]:
# Find all rows whose sentence appears more than once
dups = df[df["sentence"].duplicated(keep=False)].sort_values("sentence")
print(dups[["sentence", "label_name"]].to_string())

# For each duplicated sentence, count how many different labels it has
conflict = dups.groupby("sentence")["label_name"].nunique()
print("\nSentences with conflicting labels:", (conflict > 1).sum())

                                                                                                                                                                                                                                                                                                      sentence label_name
2395                                                                                                                                                                                                                                                  Ahlstrom 's share is quoted on the NASDAQ OMX Helsinki .    neutral
2396                                                                                                                                                                                                                                                  Ahlstrom 's share is quoted on the NASDAQ OMX Helsinki .    neutral
3093                                                      

In [15]:

import pandas as pd

# Reload from Drive so this step does not depend on earlier cells
df = pd.read_csv(ROOT / "data" / "raw" / "phrasebank_50agree.csv")
print("Before cleaning:", df.shape)

# Sentences that appear with more than one distinct label = conflicting
n_labels = df.groupby("sentence")["label"].transform("nunique")
df = df[n_labels == 1]                                  # drop ALL copies of conflicting sentences

# Remaining exact duplicates share a label, so keep the first copy only
df = df.drop_duplicates(subset="sentence", keep="first").reset_index(drop=True)
print("After cleaning:", df.shape)                      # expect (4836, 3)

Before cleaning: (4846, 3)
After cleaning: (4836, 3)


In [16]:
# Normalise: lowercase and collapse whitespace, then look for repeats
norm = df["sentence"].str.lower().str.split().str.join(" ")
print("Near-duplicates remaining:", norm.duplicated().sum())   # ideally 0; tell me if not

Near-duplicates remaining: 0


In [17]:
from sklearn.model_selection import train_test_split

# Step 1: hold out 30% of the data, keeping class proportions (stratify)
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=SEED)

# Step 2: split the held-out 30% in half: 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(part))
    print((part["label_name"].value_counts(normalize=True) * 100).round(1).to_dict())

train 3385
{'neutral': 59.4, 'positive': 28.2, 'negative': 12.5}
val 725
{'neutral': 59.4, 'positive': 28.1, 'negative': 12.4}
test 726
{'neutral': 59.4, 'positive': 28.1, 'negative': 12.5}


In [18]:
# The three parts must share no sentences at all
assert not set(train_df["sentence"]) & set(val_df["sentence"])
assert not set(train_df["sentence"]) & set(test_df["sentence"])
assert not set(val_df["sentence"]) & set(test_df["sentence"])
print("No overlap between splits")

# Save each split so every later step uses exactly the same data
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part.to_csv(ROOT / "data" / "processed" / f"phrasebank_{name}.csv", index=False)
print("Saved")

No overlap between splits
Saved


In [14]:
from datasets import load_dataset

try:
    # Same call as before, but with the full namespace/name
    ds = load_dataset("takala/financial_phrasebank", "sentences_50agree", trust_remote_code=True)
    print(ds)
except Exception as e:
    print("LOAD FAILED:", type(e).__name__)
    print(str(e)[:800])

README.md:   0%|          | 0.00/9.58k [00:00<?, ?B/s]

financial_phrasebank.py:   0%|          | 0.00/6.04k [00:00<?, ?B/s]

LOAD FAILED: HfUriError
Invalid HF URI 'hf://datasets/financial_phrasebank@main/data/FinancialPhraseBank-v1.0.zip'. Repository id must be 'namespace/name', got 'financial_phrasebank'.
